# 节点运行中抛出异常处理

因为langgraph是很大的图形结构，任何一个节点都可能出错，不能因为一个节点偶然错误，导致整个图不能执行

节点执行和容错机制大概有哪些：（对于可能出现错误，有三种容错机制）
1. 临时故障：重试（网络抖动，重新连接）
2. 超时，设置timeout（某个节点时间过长，超过时间，也判定为超时，避免阻塞）
3. 异常恢复， 节点异常后，兜底


重复计算，不能叫做容错机制，只能叫，节点优化的机制，  来添加缓存，对一个节点的结果反复使用，可以让节点触发对应的缓存，这次运行，有值，下一次条件没有变，就记录缓存下来，下一次直接使用，不需要重新计算


# 介绍前三个节点容错机制
1. 偶然重试
2. 过久超时
3. 错误兜底


先强调，现在使用的langgraph 1.1.2 ， 最多使用的重试， 要想用新的超时控制和错误兜底，要求版本大于1.2 ，很新，这里只讲不演示

理想的流程
1. 节点执行
2. 因为 `2.过久超时`或节点报错，触发节点异常
3. 走到`1. 偶然重试`， 尝试重试
4. 重试错误用完， 走到`3. 错误兜底`


因为版本限制，只有重试机制，只能延时1 2 3，去看重试的限制



In [2]:
# 重试机制
# 使用场景，临时异常，重试一下，要加retry_policy
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy
from loguru import logger
from requests import HTTPError

# 1. 声明空状态
class EmptyState(TypedDict):
    pass

# 2. 声明节点
def node_a(staet: EmptyState) -> EmptyState:
    logger.info("node_a正在运行")
    # 主动抛出异常，  模拟网络连接失败，测试重试
    raise HTTPError

# 3. 构建图
builder = StateGraph(state_schema = EmptyState)
# 添加节点的时候，要添加retry参数
builder.add_node("node_a", node_a, retry_policy=RetryPolicy(
    max_attempts=3,
    jitter=False
))

builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)

graph = builder.compile()
try: 
    graph.invoke({})
except HTTPError as e:
    logger.info("重试次数耗尽:{}", e)

2026-09-13 11:10:39.656 | INFO     | __main__:node_a:15 - node_a正在运行
2026-09-13 11:10:40.301 | INFO     | __main__:node_a:15 - node_a正在运行
2026-09-13 11:10:41.303 | INFO     | __main__:node_a:15 - node_a正在运行
2026-09-13 11:10:41.303 | INFO     | __main__:<module>:34 - 重试次数耗尽:


所有RetryPolicy配置
https://reference.langchain.com/python/langgraph，

一定要填max_attempts 默认3，最好显式指定
jitter 抖动， 默认开启， 就是默认重试，每次重试的时候，等待的时间是均匀的， 第一次0.5s， 第二次1s，第三次2s，， 最大128s， jitter就是再加随机抖动，不要均匀，有点像模拟人类，因为机器对于时间把握很精准，加上抖动，有浮动，默认添加


# 1 偶然重试
前面讲了具体的运行情况  

接下来对照参数看一遍，  下面看小的分类
1. max_attempts 最大尝试次数，包括首次执行， 第一次失败也算重试， 默认值是3， 想多重试几次，改大值就行
2. initial_interval 重试的等待间隔，默认0.5s，但不是说接下来每一次都是0.5s
3. 还有backoff_factor 重试间隔的增长倍数，有了倍数，加上上面的0.5s，就有重试的真实时间， 0.5 ，1， 2，4 （默认值2），  
4. max_interval 指数增长，后面很夸张， 所以还有最大等待时间，默认128s，如果不想超过太多，可以手动配置5s
5. jitter 抖动，更多用于并发场景，计算机对于时间精准，如果有10~20个并发，这些并发需要运行的时候，一起运行，有可能一起运行，一起失败，等待的重试时间又一样，又一起失败，可能造成瞬间的流量峰值，成功率不高，加了jitter不同的任务打散，在很小的范围内， 如果生产环境，建议保留抖动值， 如果是演示基础原理，不需要加抖动
6. 最关键retry_on， 表示哪些异常需要触发重试，很重要，但是不需要过多的配置，因为默认的重试配置写的很好，  如果想要设置置顶HTTPERROR类型， 可单个，也可以用列表制定多个，还可以加一些自定义判断的函数，  retrun的是bool类型， true要重试， false不重试(isinstance表示判断是不是httperror的实例)
    - 默认重试的条件 default_retry_on，写的很好， 默认情况对大多异常重试， 但是下面的 value type类型错误， 这些代码语法有错误，没有意义重试，更多是主流http网络访问，需要，可能服务端异常，
    - http异常里，也有一些可能不重试，4xx不重试自己参数有问题， 5xx是服务端，会重试，核心判断
    - 下面又defalut_retry_on的源代码

```python
def default_retry_on(exc: Exception) -> bool:
    import httpx
    import requests

    if isinstance(exc, ConnectionError):
        return True
    if isinstance(exc, httpx.HTTPStatusError):
        return 500 <= exc.response.status_code < 600
    if isinstance(exc, requests.HTTPError):
        return 500 <= exc.response.status_code < 600 if exc.response else True
    if isinstance( # 不重试
        exc,
        (
            ValueError,
            TypeError,
            ArithmeticError,
            ImportError,
            LookupError,
            NameError,
            SyntaxError,
            RuntimeError,
            ReferenceError,
            StopIteration,
            StopAsyncIteration,
            OSError,
        ),
    ):
        return False
    return True
```

总结一下，核心max_attempts，再3个时间自己配置， jitter生产环境打开，retry_on用默认值，一般不配置

# 2 超时控制
来看第二个错误管理，超时控制，控制单个节点单词执行耗费时间太长，导致整个卡死，这个情况，限制最大执行时间，超过这个时间，抛出异常停下来  
两个要求 1. langgrapoh >= 1.2 （比较新功能）  2. 定义的节点需要是异步节点 async def(原因是正常同步节点，一执行，线程只在节点内部，python还缺少安全机制终止同步函数，终止不了，如果是异步节点可以用异步管理方式，让节点单独抛异常，不影响其他资源（如果同步函数强行中断可能资源、锁没有释放等问题），所以这里适用于异步节点)

课上超时控制不好演示，只讲一下

## 核心思想
在add_node，添加一个timeout时间就行 ,new 一个TimeoutPolicy对象
> 前面max_interval定义的是两个节点之间超时时间，这个定义节点自己超时时间

```python
from langgraph.types import TimeoutPolicy

builder.add_node(
    "call_model",
    call_model,
    timeout=TimeoutPolicy(run_timeout=60)
)
```

配置的时候，核心参数，run_timeout= 60 超时时间60s，抛出NodeTimeoutError, 会交给RetryPolicy，  重试也不行， 最后进入错误处理

TimeoutPolicy的全部参数：
1. run_timeout: 最核心，超时时间，运行超过定义时间抛出异常（运行超时）
2. idle_timeout：空闲时间，在没有写状态，流式输出的时候，最长空闲时间 （等待空闲超时）
3. refresh_on：空闲的时候的刷新方式，手动刷新，默认自动刷新， 有需要做的事情，会进入运行时间中


官方文档：https://docs.langchain.com/oss/python/langgraph/fault-tolerance#timeouts

# 3 错误兜底

当时前面两个，重试机制+超时控制，都运行到了最后，触发了抛出异常，有抛出异常，就可以错误处理接收，也要求langgraph > = 1.2 ， 课程还是不支持

先只能用try的方式捕获， 如果版本更新了，  使用的时候， 就可以添加节点的时候， 使用`error_handler` 添加错误处理函数就行

流程
```text
节点执行失败 （包含超时控制，也是一种失败）
    ↓
是否满足 retry_policy
    ↓
满足则继续重试，直到重试次数耗尽，否则直接进行下一步
    ↓
进入 error_handler
    ↓
执行兜底恢复逻辑
```

error_handler = handle_api_error， handle_api_error函数可以自定义，

通常自定义的时候， 让他返回最新的状态更新， 也可以使用Command/goto路由到其他保底节点

错误处理适合场景： API失败兜底，远程服务到备用服务，多业务流程到补偿，不希望单个失败而终止

https://docs.langchain.com/oss/python/langgraph/fault-tolerance#error-handling